In [26]:
import pandas as pd
import re, math

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

sw_sastrawi = set(StopWordRemoverFactory().get_stop_words())

In [27]:
# Load data
CSV_PATH = '../vaksin_campak.csv'
df = pd.read_csv(CSV_PATH, sep=';')

print('Jumlah data: ', df.shape)
df.head()

Jumlah data:  (51, 1)


,komentar
0,"Melihat banyak Nakes yang menangani wabah campak, ada baiknya pertimbangkan booster Vaksin Campak terutama yang mungkin riwayat vaksinasinya ga jelas atau ga diketahui saat kecil."
1,BPOM resmi beri izin vaksin campak untuk orang dewasa.
2,Mau ngasih tau ke yang belum terpapar antivak nya si bude wellness kalau vaksin tuh untuk mengurangi penyebaran penyakit. Ih udah vaksin campak tapi tetep kena campak! tapi ga sampe meninggal kan? Banyak yang meninggal saat terpapar hanya karena belum vaksin btw aku bukan nakes
3,Waktu kecil udah pernah di vaksin campak tapi pas malam langsung demam
4,"Tingginya kasus campak, membuat banyak yg tanya ""Apakah vaksin campak bisa diberikan lebih cepat sebelum USIA 9 bulan?"" Saya akan coba jawab dari beberapa paper dan publikasi yang ada. Mari kita mulai. Sebelumnya mari kita bahas dulu, mengapa di Indonesia jadwal vaksin campak diberikan di usia 9 bulan. Bayi usia < 9 bulan, biasanya masih memiliki antibodi dari ibu yang dapat menetralisir vaksin, sehingga respon imun tubuh bayi terhadap vaksin menjadi tidak optimal."


In [28]:
def text_preprocessing(text):
    # Casefolding
    cleaned_text = text.lower()
    
    # Noise removal
    cleaned_text = re.sub(r'@\w+|#\w+', '', cleaned_text)
    cleaned_text = cleaned_text.encode('ascii', 'ignore').decode('ascii')
    cleaned_text = re.sub(r'[0-9]+', '', cleaned_text)
    cleaned_text = re.sub(r'[^\w\s]', ' ', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

    raw_tokens = cleaned_text.split()

    filtered_tokens = [t for t in raw_tokens if t not in sw_sastrawi and len(t) >= 3]

    tokens =  [stemmer.stem(t) for t in filtered_tokens]

    final_text = ' '.join(tokens)

    return cleaned_text, final_text, tokens

df[['cleaned_text', 'final_text', 'tokens']] = df['komentar'].apply(
    lambda x: pd.Series(text_preprocessing(x))
)

pd.set_option('display.max_colwidth', None)
display(df[['komentar', 'cleaned_text', 'final_text', 'tokens']].head(5))

,komentar,cleaned_text,final_text,tokens
0,"Melihat banyak Nakes yang menangani wabah campak, ada baiknya pertimbangkan booster Vaksin Campak terutama yang mungkin riwayat vaksinasinya ga jelas atau ga diketahui saat kecil.",melihat banyak nakes yang menangani wabah campak ada baiknya pertimbangkan booster vaksin campak terutama yang mungkin riwayat vaksinasinya ga jelas atau ga diketahui saat kecil,lihat banyak nakes tangan wabah campak baik timbang booster vaksin campak utama mungkin riwayat vaksinasi jelas tahu kecil,"[lihat, banyak, nakes, tangan, wabah, campak, baik, timbang, booster, vaksin, campak, utama, mungkin, riwayat, vaksinasi, jelas, tahu, kecil]"
1,BPOM resmi beri izin vaksin campak untuk orang dewasa.,bpom resmi beri izin vaksin campak untuk orang dewasa,bpom resmi beri izin vaksin campak orang dewasa,"[bpom, resmi, beri, izin, vaksin, campak, orang, dewasa]"
2,Mau ngasih tau ke yang belum terpapar antivak nya si bude wellness kalau vaksin tuh untuk mengurangi penyebaran penyakit. Ih udah vaksin campak tapi tetep kena campak! tapi ga sampe meninggal kan? Banyak yang meninggal saat terpapar hanya karena belum vaksin btw aku bukan nakes,mau ngasih tau ke yang belum terpapar antivak nya si bude wellness kalau vaksin tuh untuk mengurangi penyebaran penyakit ih udah vaksin campak tapi tetep kena campak tapi ga sampe meninggal kan banyak yang meninggal saat terpapar hanya karena belum vaksin btw aku bukan nakes,mau ngasih tau papar antivak nya bude wellness kalau vaksin tuh kurang sebar sakit udah vaksin campak tetep kena campak sampe tinggal kan banyak tinggal papar vaksin btw aku bukan nakes,"[mau, ngasih, tau, papar, antivak, nya, bude, wellness, kalau, vaksin, tuh, kurang, sebar, sakit, udah, vaksin, campak, tetep, kena, campak, sampe, tinggal, kan, banyak, tinggal, papar, vaksin, btw, aku, bukan, nakes]"
3,Waktu kecil udah pernah di vaksin campak tapi pas malam langsung demam,waktu kecil udah pernah di vaksin campak tapi pas malam langsung demam,waktu kecil udah pernah vaksin campak pas malam langsung demam,"[waktu, kecil, udah, pernah, vaksin, campak, pas, malam, langsung, demam]"
4,"Tingginya kasus campak, membuat banyak yg tanya ""Apakah vaksin campak bisa diberikan lebih cepat sebelum USIA 9 bulan?"" Saya akan coba jawab dari beberapa paper dan publikasi yang ada. Mari kita mulai. Sebelumnya mari kita bahas dulu, mengapa di Indonesia jadwal vaksin campak diberikan di usia 9 bulan. Bayi usia < 9 bulan, biasanya masih memiliki antibodi dari ibu yang dapat menetralisir vaksin, sehingga respon imun tubuh bayi terhadap vaksin menjadi tidak optimal.",tingginya kasus campak membuat banyak yg tanya apakah vaksin campak bisa diberikan lebih cepat sebelum usia bulan saya akan coba jawab dari beberapa paper dan publikasi yang ada mari kita mulai sebelumnya mari kita bahas dulu mengapa di indonesia jadwal vaksin campak diberikan di usia bulan bayi usia bulan biasanya masih memiliki antibodi dari ibu yang dapat menetralisir vaksin sehingga respon imun tubuh bayi terhadap vaksin menjadi tidak optimal,tinggi kasus campak buat banyak tanya vaksin campak beri lebih cepat usia bulan coba jawab beberapa paper publikasi mulai belum bahas dulu indonesia jadwal vaksin campak beri usia bulan bayi usia bulan biasa milik antibodi ibu menetralisir vaksin respon imun tubuh bayi vaksin jadi optimal,"[tinggi, kasus, campak, buat, banyak, tanya, vaksin, campak, beri, lebih, cepat, usia, bulan, coba, jawab, beberapa, paper, publikasi, mulai, belum, bahas, dulu, indonesia, jadwal, vaksin, campak, beri, usia, bulan, bayi, usia, bulan, biasa, milik, antibodi, ibu, menetralisir, vaksin, respon, imun, tubuh, bayi, vaksin, jadi, optimal]"


In [ ]:
def build_inverted_index(df):
    inverted_index = {}

    for doc_id, row in df.iterrows():
        tokens = row['tokens']
        
        if not tokens:
            continue

        for token in tokens:
            if token not in inverted_index:
                inverted_index[token] = {}
            inverted_index[token][str(doc_id)] = inverted_index[token].get(str(doc_id), 0) + 1
    return inverted_index

inverted_index = build_inverted_index(df)

# Preview inverted index
print(f"Jumlah kata unik di Inverted Index: {len(inverted_index)}")
print()
for word in list(inverted_index.keys())[:5]:
    print(f"Kata: '{word}' -> Muncul di: {inverted_index[word]}")

Jumlah kata unik di Inverted Index: 542

Kata: 'lihat' -> Muncul di: {'0': 1}
Kata: 'banyak' -> Muncul di: {'0': 1, '2': 1, '4': 1, '5': 1, '20': 1, '21': 1, '25': 1, '29': 1, '35': 2, '36': 1, '44': 1, '49': 1}
Kata: 'nakes' -> Muncul di: {'0': 1, '2': 1, '30': 1}
Kata: 'tangan' -> Muncul di: {'0': 1, '38': 1, '39': 2}
Kata: 'wabah' -> Muncul di: {'0': 1}


In [ ]:
def calculated_tfidf(inverted_index: dict, total_docs: int):
    tfidf_index = {}
    idf_dict = {}
    doc_length_sq = {}

    for term, doc_dict in inverted_index.items():
        df_t = len(doc_dict)
        idf = math.log10((total_docs + 1) / (df_t + 1)) + 1
        idf_dict[term] = idf
        
        tfidf_index[term] = {}
        for doc_id, raw_tf in doc_dict.items():
            log_tf = (1 + math.log10(raw_tf)) if raw_tf > 0 else 0
            weight = log_tf * idf
            tfidf_index[term][doc_id] = weight

            doc_length_sq[doc_id] = doc_length_sq.get(doc_id, 0) + weight ** 2

    doc_magnitudes = {
        doc_id: math.sqrt(sq)
        for doc_id, sq in doc_length_sq.items()
    }
    return tfidf_index, idf_dict, doc_magnitudes

# Calculate TF-IDF
total_docs = len(df)
tfidf_index, idf_dict, doc_magnitudes = calculated_tfidf(inverted_index, total_docs)

# Create TF-IDF DataFrame
tfidf_matrix = {}
for term, doc_dict in tfidf_index.items():
    for doc_id, tfidf_value in doc_dict.items():
        if doc_id not in tfidf_matrix:
            tfidf_matrix[doc_id] = {}
        tfidf_matrix[doc_id][term] = tfidf_value

# Convert to DataFrame
df_tfidf = pd.DataFrame(tfidf_matrix).fillna(0).T

# Sort
df_tfidf = df_tfidf.sort_index(key=lambda x: x.astype(int))
df_tfidf.index = [f'Komentar {int(idx) + 1}' for idx in df_tfidf.index]

display(df_tfidf.head(10))

,lihat,banyak,nakes,tangan,wabah,campak,baik,timbang,booster,vaksin,...,dengar,ruam,seluruh,serta,conjunctivitis,mata,merah,gejala,diare,ragu
Komentar 1,2.414973,1.60206,2.113943,2.113943,2.414973,1.301030,1.937852,2.414973,2.238882,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 2,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 3,0.000000,1.60206,2.113943,0.000000,0.000000,1.301030,0.000000,0.000000,0.000000,1.477121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 4,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 5,0.000000,1.60206,0.000000,0.000000,0.000000,1.477121,0.000000,0.000000,0.000000,1.602060,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 6,0.000000,1.60206,0.000000,0.000000,0.000000,1.301030,0.000000,0.000000,0.000000,1.301030,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 7,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 8,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 9,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Komentar 10,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
def search(query: str, tfidf_index: dict, idf_dict: dict, doc_magnitudes: dict, doc_lookup: dict):
    _, _, query_tokens = text_preprocessing(query)
    if not query_tokens:
        return []
    
    query_raw_tf = {}
    for token in query_tokens:
        query_raw_tf[token] = query_raw_tf.get(token, 0) + 1

    query_tfidf = {}
    query_length_sq = 0.0

    for term, raw_tf in query_raw_tf.items():
        if term not in idf_dict:
            continue
        log_tf = (1 + math.log10(raw_tf)) if raw_tf > 0 else 0
        weight = log_tf * idf_dict[term]
        query_tfidf[term] = weight
        query_length_sq += weight ** 2

    if query_length_sq == 0:
        return []
    
    query_magnitude = math.sqrt(query_length_sq)

    dot_products = {}
    for term, q_weight in query_tfidf.items():
        if term in tfidf_index:
            for doc_id, d_weight in tfidf_index[term].items():
                dot_products[doc_id] = dot_products.get(doc_id, 0) + (q_weight * d_weight)

    if not dot_products:
        return []

    scores = []
    for doc_id, dot_prod in dot_products.items():
        denom = query_magnitude * doc_magnitudes.get(doc_id, 1.0)
        if denom == 0:
            continue
        similarity = dot_prod / denom
        
        row_data = doc_lookup[doc_id]
        scores.append({
            "doc_id": doc_id,
            "komentar": row_data['komentar'],
            "score": round(similarity, 4)
        })

    scores = sorted(scores, key=lambda x: x['score'], reverse=True)
    return scores

# Create doc_lookup
doc_lookup = {}
for doc_id, row in df.iterrows():
    doc_lookup[str(doc_id)] = {
        'komentar': row['komentar']
    }

# Test search
query = "vaksin campak"
results = search(query, tfidf_index, idf_dict, doc_magnitudes, doc_lookup)

print(f"Query: '{query}'")
print(f"Jumlah hasil: {len(results)}\n")

for i, result in enumerate(results[:5], 1):
    print(f"{i}. Doc {result['doc_id']} (Score: {result['score']})")
    print(f"   Komentar: {result['komentar'][:100]}...")
    print()

Query: 'vaksin campak'
Jumlah hasil: 51

1. Doc 9 (Score: 0.4032)
   Komentar: Vaksin Campak tersedia di imuni....

2. Doc 13 (Score: 0.3913)
   Komentar: Inilah pentingnya vaksin campak agar tidak dicampakkan (diisolasi)...

3. Doc 18 (Score: 0.334)
   Komentar: Udah vaksin campak dari kecil tapi pas dewasa ternyata aku masih dicampakkan. (???)...

4. Doc 6 (Score: 0.2999)
   Komentar: ANAK AKU GA VAKSIN CAMPAK SEHAT SEHAT AJA TUH...

5. Doc 1 (Score: 0.2724)
   Komentar: BPOM resmi beri izin vaksin campak untuk orang dewasa....

